In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura exibição de gráficos inline no notebook
%matplotlib inline

# Ajustar scikit-learn à tabela de características reais salva

Execute o tutorial 40 primeiro com o mesmo ``EEGDASH_CACHE_DIR``. Esta página lê sua tabela
real nm000118 e o esquema correspondente, treina nos sujeitos 1 e 2 e, em seguida, testa no sujeito 3.
Ela não baixa os sinais novamente nem substitui uma tabela ausente.
A fonte com três participantes possui cerca de 21.1 MB; consulte o tutorial 40 para proveniência
dos dados e extração. Esta é uma pequena demonstração de fluxo de trabalho.

## Antes de começar
Instale as dependências do EEGDash e execute ``plot_40_first_features.py`` primeiro.
Use exatamente o mesmo ``EEGDASH_CACHE_DIR`` para ambos os scripts; o padrão ``.eegdash_cache``
é relativo ao diretório de trabalho. Você precisa do CSV e do seu esquema JSON adjacente,
mas não é necessário nenhum download de sinal em tempo real para esta página.
O tutorial 11 explica a divisão por participantes usada abaixo.


In [ ]:
# Importa módulos para leitura de JSON e manipulação de arquivos no sistema
import json
import os
from pathlib import Path

# Importa bibliotecas para plotagem, computação numérica e tabelas de dados
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa regressor logístico, exibição de matriz de confusão e métricas do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
# Importa classes para montagem de pipeline e escalonamento padrão de dados
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Carregar a tabela exata gravada pelo tutorial 40
A lista de características do esquema é uma lista de permissões (*allowlist*) explícita.
Selecionar todas as colunas numéricas vazaria a resposta por meio de ``target`` ou ``frequency_hz``
e também poderia incluir identificadores de amostra como preditores. A matriz resultante possui
uma linha por ensaio gravado e 24 colunas de banda/canal; os vetores alvo e de sujeito
são mantidos separados.

Os valores armazenados são resumos de potência linear. O Log10 comprime seu intervalo antes
do aprendizado, com um piso numérico para potência zero. Essa transformação atua em cada
valor de forma independente; ao contrário do StandardScaler, ela não estima estatísticas
a partir de participantes retidos (*held-out*). A tabela cruzada impressa verifica a cobertura
das classes após a transferência do arquivo.



In [ ]:
# Define o caminho do diretório de cache
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Define o caminho para a tabela gerada no tutorial 40
path = cache_dir / "plot_40_features.csv"
# Verifica se os arquivos CSV e JSON correspondentes existem no cache
if not path.exists() or not path.with_suffix(".json").exists():
    raise FileNotFoundError(
        "Run plot_40_first_features.py with the same EEGDASH_CACHE_DIR first"
    )
# Lê o arquivo CSV carregando os identificadores explicitamente como strings
table = pd.read_csv(path, dtype={"subject": str, "session": str, "run": str})
# Carrega o esquema JSON que documenta as colunas de características oficiais e metadados
schema = json.loads(path.with_suffix(".json").read_text())
# Valida o identificador do conjunto de dados no esquema
assert schema["dataset"] == "nm000118"
# Obtém a lista autoritativa de colunas de características
columns = schema["feature_columns"]
# Valida que as colunas de características estão presentes no CSV
assert columns and set(columns).issubset(table.columns)
# Garante ausência de duplicatas para cada ensaio registrado
assert not table.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Aplica escala log10 com piso mínimo de 1e-30 para formar a matriz de preditores X
X = np.log10(np.maximum(table[columns].to_numpy(), 1e-30))
# Extrai o array numérico das classes alvo y
y = table["target"].to_numpy(dtype=int)
# Extrai o array de identificadores dos sujeitos como grupos
groups = table["subject"].astype(str).to_numpy()
# Valida finitude numérica de X e presença exata dos três sujeitos
assert np.isfinite(X).all()
assert set(groups) == {"1", "2", "3"}
# Exibe tabela cruzada de contagem de classes por sujeito
print(pd.crosstab(groups, y))
# Exibe as dimensões da matriz de características gerada
print("Feature matrix:", X.shape)

## 2. Ajustar o escalador e o classificador apenas nos sujeitos de treino
Os sujeitos 1 e 2 fornecem todas as estatísticas do escalador e coeficientes do classificador;
o sujeito 3 fornece apenas previsões finais. O pipeline vincula essas operações aprendidas
para que ``predict`` aplique a escala do treino em vez de ajustar uma nova escala ao participante de teste.
O algoritmo de otimização possui até 1.000 iterações; se avisos de convergência ocorrerem, inspecione
o condicionamento do treino e otimize dentro dos dados de treino antes de tratar os coeficientes
ajustados como estáveis.

A acurácia balanceada é a média da sensibilidade ao longo das doze classes de frequência.
Um resultado próximo de 1/12 pode ser consistente com as informações descartadas por bandas largas:
muitas frequências distintas de cintilação (*flicker*) caem na mesma banda. Essa é uma limitação
dessa representação de características, não um motivo para substituir a pontuação medida por
um número mais atraente.



In [ ]:
# Define as máscaras booleanas para divisão entre treino (sujeitos 1 e 2) e teste (sujeito 3)
train, test = groups != "3", groups == "3"
# Valida que os grupos de treino e teste são estritamente disjuntos
assert set(groups[train]).isdisjoint(groups[test])
# Valida que todas as 12 classes estão presentes tanto no treino quanto no teste
assert set(y[train]) == set(y[test]) == set(schema["mapping"].values())
# Constrói o pipeline contendo escalonamento padronizado e regressão logística com até 1000 iterações
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# Ajusta o pipeline estritamente nos dados de treino dos sujeitos 1 e 2
pipe.fit(X[train], y[train])
# Realiza predições para as amostras não vistas do sujeito 3
predictions = pipe.predict(X[test])
# Calcula e exibe a acurácia balanceada obtida para o sujeito 3
print("Subject 3 balanced accuracy:", balanced_accuracy_score(y[test], predictions))

## 3. Inspecionar as previsões reais e os coeficientes ajustados
A matriz de confusão é normalizada pelas linhas, portanto cada linha descreve a distribuição
de classes previstas para uma única classe verdadeira. Seus rótulos inteiros são os índices
de frequência armazenados no mapeamento JSON.

Para cada característica, o gráfico da direita calcula a média do valor absoluto do coeficiente
ajustado em todas as decisões de classe e exibe os oito maiores. O escalonamento torna as
magnitudes mais fáceis de comparar, mas características correlacionadas podem compartilhar
ou trocar pesos. O uso de valores absolutos também remove a direção e o sinal específico da classe,
de modo que o gráfico é um resumo do modelo, não uma evidência de que uma banda causa a resposta ao estímulo.



In [ ]:
# Cria figura com dois gráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
# Plota matriz de confusão normalizada por linha para o sujeito retido 3
ConfusionMatrixDisplay.from_predictions(
    y[test],
    predictions,
    normalize="true",
    include_values=False,
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Held-out subject 3 (frequency-class indices)")
# Calcula a média dos coeficientes absolutos do regressor logístico para cada característica
weights = np.abs(pipe.named_steps["logisticregression"].coef_).mean(axis=0)
# Identifica os índices dos 8 coeficientes com maior magnitude
order = np.argsort(weights)[-8:]
# Plota gráfico de barras horizontais com as 8 características mais influentes
axes[1].barh(np.asarray(columns)[order], weights[order])
# Configura rótulos e títulos do gráfico de coeficientes
axes[1].set(
    xlabel="Mean absolute standardized coefficient", title="Training-fit coefficients"
)
# Exibe a figura gerada
plt.show()

## Escolher o próximo experimento de características
O tutorial 12 retém intervalos espectrais finos (*fine bins*) em vez de somas em bandas largas.
Compare essa ideia em uma divisão de validação de treino antes de abrir uma nova coorte de teste.
Se alterar a tabela no tutorial 40, gere novamente seu esquema e execute esta página novamente
em vez de tentar adivinhar manualmente a ordem das colunas.

